# Praktikum 3 - Klasifikasi Teks dengan Multinomial Naive Bayes (SMS Spam)

**Mata Kuliah:** Pembelajaran Mesin
**Jobsheet 09:** Klasifikasi

### Tujuan Praktikum:
1. Memuat dan membersihkan dataset teks SMS Spam (`spam.csv`).
2. Melakukan ekstraksi fitur teks Bag-of-Words menggunakan `CountVectorizer`.
3. Membangun model `MultinomialNB` untuk membedakan pesan sah (*ham*) dan pesan sampah (*spam*).
4. Mengevaluasi performa model pada data latih dan data uji.

## Langkah 1 - Load Data SMS Spam

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

data_path = r'..\Dataset\spam.csv'

df = pd.read_csv(data_path, encoding='latin-1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


## Langkah 2 - Pra Pengolahan Data Teks

In [2]:
# Drop kolom un-named yang tidak terpakai
df = df.drop(columns=[col for col in ['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'] if col in df.columns])

# Rename kolom menjadi lebih deskriptif
df = df.rename(columns={'v1': 'Labels', 'v2': 'SMS'})

# Encoding Label: 'ham' -> 0, 'spam' -> 1
df['Labels'] = df['Labels'].map({'ham': 0, 'spam': 1})

print('Dimensi data:', df.shape)
print('Distribusi kelas label (0=ham, 1=spam):\n', df['Labels'].value_counts())
df.head()

Dimensi data: (5572, 2)
Distribusi kelas label (0=ham, 1=spam):
 Labels
0    4825
1     747
Name: count, dtype: int64


,Labels,SMS
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


## Langkah 3 - Ekstraksi Fitur Teks dengan CountVectorizer

In [3]:
X = df['SMS'].to_numpy(dtype=str)
y = df['Labels'].to_numpy(dtype=int)

# Split data training dan testing (80:20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50, stratify=y)

# Ekstraksi fitur Bag of Words
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

print(f'Total kosakata unik (*vocabulary*): {len(bow.get_feature_names_out())}')
print(f'Dimensi matriks X_train_bow: {X_train_bow.shape}')

Total kosakata unik (*vocabulary*): 7684
Dimensi matriks X_train_bow: (4457, 7684)


## Langkah 4 - Pelatihan dan Evaluasi Model Multinomial Naive Bayes

In [4]:
mnb = MultinomialNB()
mnb.fit(X_train_bow, y_train)

y_pred_train = mnb.predict(X_train_bow)
y_pred_test = mnb.predict(X_test_bow)

acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)

print(f'Akurasi Data Latih : {acc_train:.4f} ({acc_train*100:.2f}%)')
print(f'Akurasi Data Uji   : {acc_test:.4f} ({acc_test*100:.2f}%)')
print('\nLaporan Klasifikasi Data Uji:\n', classification_report(y_test, y_pred_test, target_names=['Ham', 'Spam']))

Akurasi Data Latih : 0.9933 (99.33%)
Akurasi Data Uji   : 0.9883 (98.83%)

Laporan Klasifikasi Data Uji:
               precision    recall  f1-score   support

         Ham       0.99      1.00      0.99       966
        Spam       0.99      0.93      0.96       149

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115

